#### 3.1.2 Проверка и очистка

**Входы**

- Исходные данные

**Выходы**

- Данные без полных повторов и явных ошибок
- Решения о том, что убрать или исправить

**Что делаем**

Проверим одинаковые строки, подозрительные значения и согласованность связанных полей

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


##### Проверим данные

**Дубликаты** — полностью одинаковые строки. Посчитаем, сколько таких повторов есть и сколько строк останется, если оставить по одному экземпляру

Отдельно проверим, встречаются ли одинаковые признаки с разным `loan_status`: это уже не полное совпадение записей

In [6]:
duplicate_extra = data.duplicated()
duplicate_all = data.duplicated(keep=False)
feature_columns = data.columns.drop('loan_status')
conflicting_groups = (
    data.groupby(list(feature_columns), dropna=False)['loan_status']
        .nunique().gt(1)
)

duplicate_summary = pd.DataFrame({
    'Количество': [
        int(duplicate_extra.sum()),
        int(duplicate_all.sum()),
        len(data.loc[duplicate_all].drop_duplicates()),
        int(conflicting_groups.sum()),
    ]
}, index=[
    'Повторные экземпляры сверх первого',
    'Все строки в повторяющихся группах',
    'Группы полностью одинаковых строк',
    'Группы одинаковых признаков с разными loan_status',
])
display(duplicate_summary)
display(data.loc[duplicate_all].sort_values(list(data.columns)).head(10))


,Количество
Повторные экземпляры сверх первого,165
Все строки в повторяющихся группах,330
Группы полностью одинаковых строк,165
Группы одинаковых признаков с разными loan_status,0


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
15944,21,8088,RENT,NaN,MEDICAL,C,1200,15.23,0,0.15,Y,2
16835,21,8088,RENT,NaN,MEDICAL,C,1200,15.23,0,0.15,Y,2
2431,21,15600,RENT,0,MEDICAL,A,2800,7.4,1,0.18,N,4
17758,21,15600,RENT,0,MEDICAL,A,2800,7.4,1,0.18,N,4
2498,21,18000,RENT,0,DEBTCONSOLIDATION,A,3000,7.9,1,0.17,N,2
17601,21,18000,RENT,0,DEBTCONSOLIDATION,A,3000,7.9,1,0.17,N,2
1778,21,20000,RENT,0,MEDICAL,A,2000,NaN,0,0.1,N,3
17642,21,20000,RENT,0,MEDICAL,A,2000,NaN,0,0.1,N,3
740,21,21600,OWN,NaN,VENTURE,A,7125,6.99,0,0.33,N,3
17283,21,21600,OWN,NaN,VENTURE,A,7125,6.99,0,0.33,N,3


##### Проверим, согласуются ли значения между собой

Например, стаж не должен быть длиннее жизни человека, а доля кредита от дохода должна быть близка к отношению суммы кредита к доходу

Ниже — рабочие границы проверки. Возраст 100 лет и начало работы в 14 лет — наши допущения для поиска подозрительных записей, а не правила автора датасета. Если нужное поле пропущено, такую проверку для строки пропускаем

| Что проверяем | Условие | Почему |
| --- | --- | --- |
| Возраст | Не больше 100 лет | Более высокие значения требуют проверки |
| Стаж | Не больше 86 лет | 100 лет минус предполагаемый возраст начала работы 14 лет |
| Стаж и возраст | Стаж ≤ возраст − 14 | Предполагаем, что человек начал работать не раньше 14 лет |
| Кредитная история и возраст | История ≤ возраст − 16 | Предполагаем начало кредитной истории не раньше 16 лет |
| Доля кредита от дохода | Отличие от суммы / дохода не больше 0,005 | Учитываем округление доли до сотых |
| Возраст, стаж, кредитная история | Целые значения | Проверяем, записаны ли длительности в полных годах; дробный стаж сам по себе не был бы ошибкой |

In [7]:
ratio_expected = data['loan_amnt'] / data['person_income']
ratio_difference = (data['loan_percent_income'] - ratio_expected).abs()

checks = {
    'Целочисленность возраста': (
        ['person_age'], data['person_age'] % 1 != 0),
    'Целочисленность стажа': (
        ['person_emp_length'], data['person_emp_length'] % 1 != 0),
    'Целочисленность кредитной истории': (
        ['cb_person_cred_hist_length'], data['cb_person_cred_hist_length'] % 1 != 0),
    'Возраст ≤ 100': (
        ['person_age'], data['person_age'] > 100),
    'Стаж ≤ 86': (
        ['person_emp_length'], data['person_emp_length'] > 86),
    'Стаж ≤ возраст − 14': (
        ['person_age', 'person_emp_length'],
        data['person_emp_length'] > data['person_age'] - 14),
    'Кредитная история ≤ возраст − 16': (
        ['person_age', 'cb_person_cred_hist_length'],
        data['cb_person_cred_hist_length'] > data['person_age'] - 16),
    'Доля кредита соответствует сумме / доходу (погрешность ≤ 0,005)': (
        ['loan_percent_income', 'loan_amnt', 'person_income'],
        ratio_difference > 0.005 + 1e-9),
}

constraint_results = []
constraint_violations = pd.DataFrame(index=data.index)
for name, (columns, violation) in checks.items():
    available = data[columns].notna().all(axis=1)
    # Согласованность проверяем только при допустимом абсолютном стаже.
    if name == 'Стаж ≤ возраст − 14':
        available &= data['person_emp_length'] <= 86
    if 'person_income' in columns:
        available &= data['person_income'] > 0
    constraint_violations[name] = available & violation
    constraint_results.append({
        'Проверка': name,
        'Проверено': int(available.sum()),
        'Не проверено': int((~available).sum()),
        'Нарушений': int(constraint_violations[name].sum()),
    })

display(pd.DataFrame(constraint_results).set_index('Проверка').rename_axis(None))


,Проверено,Не проверено,Нарушений
Целочисленность возраста,32581,0,0
Целочисленность стажа,31686,895,0
Целочисленность кредитной истории,32581,0,0
Возраст ≤ 100,32581,0,5
Стаж ≤ 86,31686,895,2
Стаж ≤ возраст − 14,31684,897,0
Кредитная история ≤ возраст − 16,32581,0,0
"Доля кредита соответствует сумме / доходу (погрешность ≤ 0,005)",32581,0,429


##### Очистим найденные ошибки

Убираем полные повторы и записи с недопустимым возрастом. Недопустимый стаж заменяем пропуском; способ заполнения пропусков выберем позже

Исходная таблица остаётся в `data_raw`, а с очищенной работаем через `data`

In [ ]:
# Каждый запуск повторяет очистку исходных данных
data = data_raw.drop_duplicates().copy()
invalid_age = data['person_age'] > 100
data = data.loc[~invalid_age].copy()

invalid_employment = data['person_emp_length'] > 86
data.loc[invalid_employment, 'person_emp_length'] = float('nan')


#### Выводы и решения после проверки корректности

In [ ]:
conclusions = update_findings(conclusions, [
    ('f03', 'В person_emp_length (2,75 % записей) и в loan_int_rate (9.56% записей) есть пропуски', 'Способ обработки пропусков выберем после анализа распределений'),
    ('f07', 'Найдено 165 групп ровно по две полностью одинаковые строки: 330 записей, из них 165 повторных (0,51 % выборки).\nВозможно, записи повторились при подготовке данных', 'Убрать полные повторы, оставив по одной записи'),
    ('f08', 'Возраст превышает рабочую границу 100 лет в 5 записях: 144 года в трёх записях, 123 года в двух', 'Убрать эти 5 записей: их мало, а надёжно восстановить возраст нельзя'),
    ('f09', 'В двух записях стаж равен 123 годам и превышает выбранную границу 86 лет', 'Заменить недопустимый стаж пропусками, сохранив записи'),
    ('f10', 'Для записей со стажем ≤ 86 лет выполняется условие «стаж ≤ возраст − 14»', '—'),
    ('f11', 'Выполняется условие «кредитная история ≤ возраст − 16»', '—'),
    ('f12', 'В 429 записях (1,32 % всей выборки) доля кредита отличается от отношения суммы к доходу более чем на 0,005; одним округлением до сотых эти расхождения не объясняются', '—'),
    ('f13', 'Возраст, стаж и длина кредитной истории везде, где они указаны, записаны целыми числами', '—'),
], remove=['f05'])
display(findings_view(conclusions))